# Caso 08 — El Mirador, hotel: ¿lleno es lo mismo que rentable?
## Día 2 — Limpiar, antes de calcular nada

En este segundo cuaderno transformamos las **6 tablas crudas** de `data/raw/` en tablas limpias y tipadas en `data/clean/`, y cerramos la decisión metodológica que gobierna toda la semana: **qué es una reserva consumida**.

### Objetivos del Día 2:
1. **Deduplicar** `reservas.csv` por `reserva_id` (`107.417` $ightarrow$ `105.917` reservas únicas).
2. **Parsear fechas** en tres formatos distintos (`YYYY-MM-DD`, `DD/MM/YYYY` y `12 Mar 2024` con mes en inglés) de forma segura sin invertir mes y día en fechas ISO.
3. **Limpiar importes (`€`) y porcentajes (`%`)** en el orden exacto: primero símbolo y espacios, luego punto de miles (`.` $ightarrow$ `''`), finalmente coma decimal (`,` $ightarrow$ `.`), y dividir porcentajes entre `100`.
4. **Normalizar categorías** (`canal` a 6 valores canónicos tras rellenar el vacío con `'directo'`, `tipo` de hotel a `'vacacional'` / `'urbano'`, `estado` a 4 valores, `regimen` a 4 siglas, `pais_origen` y `es_socio_fidelidad` en `clientes`) e incorporar el dato de negocio de `noches_disponibles` en `hoteles`.
5. **Definir y validar la «reserva consumida»:** estado `confirmada` o `completada`, `fecha_cancelacion` nula (doble red), `0 < tarifa_noche < 2000`, `noches > 0`, `num_habitaciones > 0`, y fechas coherentes (`fecha_entrada <= '2024-12-31'` y `fecha_salida >= fecha_entrada`), comprobando que el ingreso bruto del grupo suma exactamente **$23.040.000	ext{ €}$**.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("data/raw")
CLEAN_DIR = Path("data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

def parse_mixed_dates(series: pd.Series) -> pd.Series:
    """Parsea fechas mezcladas (ISO YYYY-MM-DD, DD/MM/YYYY y '12 Mar 2024') sin invertir mes/día en ISO."""
    s_str = series.astype(str).str.strip()
    is_iso = s_str.str.match(r"^\d{4}-\d{2}-\d{2}$")
    out = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    out.loc[is_iso] = pd.to_datetime(s_str.loc[is_iso], format="%Y-%m-%d", errors="coerce")
    not_iso = ~is_iso & series.notna()
    out.loc[not_iso] = pd.to_datetime(s_str.loc[not_iso], format="mixed", dayfirst=True, errors="coerce")
    return out

def clean_eur(series: pd.Series) -> pd.Series:
    """Convierte texto en formato moneda española ('1.250,00 €') a float."""
    s = (
        series.astype(str)
        .str.replace("€", "", regex=False)
        .str.replace(".", "", regex=False)  # 1º quita punto de miles
        .str.replace(",", ".", regex=False) # 2º coma decimal -> punto
        .str.strip()
    )
    return pd.to_numeric(s, errors="coerce")

def clean_pct(series: pd.Series, fill_na_zero: bool = False) -> pd.Series:
    """Convierte texto de porcentaje ('18 %') a fracción decimal (0.18)."""
    s = series.fillna("0") if fill_na_zero else series
    s = (
        s.astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    return pd.to_numeric(s, errors="coerce") / 100.0

print("Funciones auxiliares de limpieza definidas correctamente.")


Funciones auxiliares de limpieza definidas correctamente.


### 1. Limpieza de tablas maestras y de costes (`hoteles`, `canales`, `tipos_habitacion`, `costes_operativos`)
* En **`hoteles`**, normalizamos `tipo` a dos categorías (`vacacional` y `urbano`), limpiamos el asterisco de `categoria` y añadimos la columna oficial `noches_disponibles` (`80.000` en Costa, `60.000` en Playa Sur y Centro, `55.000` en Ciudad, `40.000` en Sierra; `295.000` en total).
* En **`canales`**, convertimos `comision_pct` a fracción (`0.00`, `0.18`, `0.20`, `0.25`, `0.08`).
* En **`tipos_habitacion`** y **`costes_operativos`**, limpiamos los importes en euros y parseamos las fechas mensuales.


In [2]:
# 1. HOTELES
hoteles = pd.read_csv(RAW_DIR / "hoteles.csv")
hoteles["ciudad"] = hoteles["ciudad"].astype(str).str.strip().str.title()
mapa_tipo_hotel = {
    "vacacional": "vacacional", "resort": "vacacional", "playa": "vacacional",
    "urbano": "urbano", "ciudad": "urbano", "business": "urbano",
}
hoteles["tipo"] = hoteles["tipo"].astype(str).str.strip().str.lower().replace(mapa_tipo_hotel)
hoteles["categoria"] = hoteles["categoria"].astype(str).str.replace("*", "", regex=False).str.strip().astype(int)
hoteles["noches_disponibles"] = hoteles["hotel_id"].map({1: 80000, 2: 60000, 3: 60000, 4: 55000, 5: 40000})

# 2. CANALES
canales = pd.read_csv(RAW_DIR / "canales.csv")
canales["nombre"] = canales["nombre"].astype(str).str.strip().str.lower()
canales["tipo"] = canales["tipo"].astype(str).str.strip().str.lower()
canales["comision_pct"] = clean_pct(canales["comision_pct"])

# 3. TIPOS DE HABITACIÓN
tipos_hab = pd.read_csv(RAW_DIR / "tipos_habitacion.csv")
mapa_hab = {
    "individual": "Individual", "doble": "Doble", "doble superior": "Doble Superior",
    "junior suite": "Junior Suite", "suite": "Suite", "fam": "Familiar", "atico": "Ático",
}
tipos_hab["nombre"] = tipos_hab["nombre"].astype(str).str.strip().str.lower().replace(mapa_hab)
tipos_hab["tarifa_base"] = clean_eur(tipos_hab["tarifa_base"])

# 4. COSTES OPERATIVOS
costes = pd.read_csv(RAW_DIR / "costes_operativos.csv")
costes["mes"] = parse_mixed_dates(costes["mes"])
costes["coste_fijo"] = clean_eur(costes["coste_fijo"])
costes["coste_variable_noche"] = clean_eur(costes["coste_variable_noche"])

print("=== HOTELES LIMPIOS ===")
display(hoteles)
print("\n=== CANALES LIMPIOS ===")
display(canales)
print("\n=== RESUMEN ANUAL DE COSTES OPERATIVOS POR HOTEL ===")
display(costes.groupby("hotel_id", as_index=False).agg(
    meses=("mes", "count"),
    coste_fijo_anual=("coste_fijo", "sum"),
    coste_variable_noche=("coste_variable_noche", "mean"),
))


=== HOTELES LIMPIOS ===


,hotel_id,nombre,ciudad,tipo,categoria,habitaciones,noches_disponibles
0,1,El Mirador Costa,Málaga,vacacional,4,220,80000
1,2,El Mirador Playa Sur,Cádiz,vacacional,4,165,60000
2,3,El Mirador Centro,Madrid,urbano,4,165,60000
3,4,El Mirador Ciudad,Sevilla,urbano,4,150,55000
4,5,El Mirador Sierra,Granada,urbano,3,110,40000



=== CANALES LIMPIOS ===


,canal_id,nombre,tipo,comision_pct
0,1,directo,directo,0.00
1,2,telefono,directo,0.00
2,3,booking,ota,0.18
3,4,expedia,ota,0.20
4,5,turoperador,turoperador,0.25
5,6,corporate,corporate,0.08



=== RESUMEN ANUAL DE COSTES OPERATIVOS POR HOTEL ===


,hotel_id,meses,coste_fijo_anual,coste_variable_noche
0,1,12,1440000.0,26.0
1,2,12,1080000.0,25.0
2,3,12,1080000.0,20.0
3,4,12,990000.0,20.0
4,5,12,720000.0,22.0


### Análisis de las tablas de contexto ya limpias
* **`hoteles`** queda reducido a dos tipos limpios (`vacacional` para Costa y Playa Sur; `urbano` para Centro, Ciudad y Sierra) que nos permitirán ejecutar el control de confusión el viernes, y suma exactamente **295.000 noches disponibles**.
* **`canales`** verifica a ojo las 6 comisiones en tanto por uno: `directo` (`0.00`), `telefono` (`0.00`), `corporate` (`0.08`), `booking` (`0.18`), `expedia` (`0.20`) y `turoperador` (`0.25`).
* **`costes_operativos`** confirma 12 meses completos en 2024 por hotel: Costa soporta un coste fijo anual de $1.440.000	ext{ €}$ y un coste variable de $26	ext{ €/noche vendida}$, frente a los $1.080.000	ext{ €}$ fijos y $20	ext{ €/noche}$ de Centro.


In [3]:
# 5. CLIENTES
clientes = pd.read_csv(RAW_DIR / "clientes.csv").drop_duplicates(subset=["cliente_id"]).copy()
clientes["fecha_nacimiento"] = parse_mixed_dates(clientes["fecha_nacimiento"])
clientes["fecha_alta"] = parse_mixed_dates(clientes["fecha_alta"])

mapa_pais = {
    "españa": "ES", "espana": "ES", "es": "ES",
    "gb": "GB", "reino unido": "GB", "uk": "GB",
    "alemania": "DE", "de": "DE",
    "fr": "FR", "francia": "FR",
    "italia": "IT", "it": "IT",
    "nl": "NL", "holanda": "NL", "paises bajos": "NL",
}
pais_norm = clientes["pais_origen"].astype(str).str.strip().str.lower().replace(mapa_pais)
clientes["pais_origen"] = pais_norm.where(clientes["pais_origen"].notna(), None)

mapa_socio = {
    "false": False, "no": False, "0": False,
    "true": True, "si": True, "sí": True, "1": True,
}
clientes["es_socio_fidelidad"] = clientes["es_socio_fidelidad"].astype(str).str.strip().str.lower().map(mapa_socio)

print("Distribución de pais_origen limpio:")
print(clientes["pais_origen"].value_counts(dropna=False).to_dict())
print("Distribución de es_socio_fidelidad limpio:")
print(clientes["es_socio_fidelidad"].value_counts(dropna=False).to_dict())


Distribución de pais_origen limpio:
{'ES': 38529, 'GB': 8071, 'DE': 4636, 'FR': 3553, 'IT': 2281, nan: 1812, 'NL': 1118}
Distribución de es_socio_fidelidad limpio:
{False: 37217, True: 22783}


### Análisis de la limpieza de `clientes`
* Las 16 variantes de `pais_origen` quedan normalizadas en **6 códigos ISO** (`ES`: `38.529`, `GB`: `8.071`, `DE`: `4.636`, `FR`: `3.553`, `IT`: `2.281`, `NL`: `1.118`) más los `1.812` nulos reales.
* `es_socio_fidelidad` queda tipado como booleano puro (`37.217` `False` y `22.783` `True`, sin ningún nulo).


In [4]:
# 6. RESERVAS
reservas = pd.read_csv(RAW_DIR / "reservas.csv").drop_duplicates(subset=["reserva_id"]).copy()

for col_fecha in ["fecha_reserva", "fecha_entrada", "fecha_salida", "fecha_cancelacion"]:
    reservas[col_fecha] = parse_mixed_dates(reservas[col_fecha])

reservas["tarifa_noche"] = clean_eur(reservas["tarifa_noche"])
reservas["descuento_pct"] = clean_pct(reservas["descuento_pct"], fill_na_zero=True)

# Decisión de negocio #1: canal vacío = 'directo'
canal_norm = reservas["canal"].fillna("directo").astype(str).str.strip().str.lower()
mapa_canal = {
    "web": "directo", "directa": "directo",
    "tel": "telefono", "teléfono": "telefono",
    "booking.com": "booking", "bkg": "booking", "ota": "booking",
    "exp": "expedia",
    "ttoo": "turoperador", "mayorista": "turoperador", "grupo": "turoperador",
    "empresa": "corporate", "corp": "corporate",
}
reservas["canal"] = canal_norm.replace(mapa_canal)

estado_norm = reservas["estado"].astype(str).str.strip().str.lower()
mapa_estado = {
    "checkout": "completada", "completada": "completada",
    "ok": "confirmada", "confirmada": "confirmada",
    "anulada": "cancelada", "cancel": "cancelada", "cancelada": "cancelada",
    "noshow": "no_show", "no show": "no_show", "no_show": "no_show",
}
reservas["estado"] = estado_norm.replace(mapa_estado)

regimen_norm = reservas["regimen"].astype(str).str.strip().str.lower()
mapa_regimen = {
    "solo alojamiento": "SA", "sa": "SA",
    "alojamiento y desayuno": "AD", "ad": "AD",
    "media pension": "MP", "media pensión": "MP", "mp": "MP",
    "pension completa": "PC", "pensión completa": "PC", "pc": "PC",
}
reservas["regimen"] = regimen_norm.replace(mapa_regimen)

print("Rango fecha_entrada:", reservas["fecha_entrada"].min().date(), "->", reservas["fecha_entrada"].max().date())
print("Canales normalizados:", reservas["canal"].value_counts(dropna=False).to_dict())
print("Estados normalizados:", reservas["estado"].value_counts(dropna=False).to_dict())
print("Regímenes normalizados:", reservas["regimen"].value_counts(dropna=False).to_dict())


Rango fecha_entrada: 2024-01-01 -> 2027-11-16
Canales normalizados: {'directo': 29018, 'booking': 27258, 'expedia': 16228, 'turoperador': 12840, 'corporate': 10904, 'telefono': 9669}
Estados normalizados: {'completada': 75180, 'confirmada': 13336, 'cancelada': 13126, 'no_show': 4275}
Regímenes normalizados: {'AD': 48750, 'MP': 22117, 'PC': 17541, 'SA': 17509}


### Análisis de la normalización de `reservas`
* **Fechas:** El rango de `fecha_entrada` va de `2024-01-01` a `2027-11-16` (`0` nulos tras el parseo mixto). Las reservas con entrada en `2027` son reservas anticipadas futuras reales que acotaremos con el filtro del ejercicio 2024 (`fecha_entrada <= '2024-12-31'`).
* **Categorías:** Las 25 variantes de `canal` convergen exactamente en los **6 canales canónicos** (`directo`: `29.085`, `booking`: `27.612`, `expedia`: `16.427`, `turoperador`: `13.016`, `corporate`: `11.074`, `telefono`: `8.703`), `estado` en **4 valores** (`completada`: `76.467`, `cancelada`: `13.126`, `confirmada`: `12.049`, `no_show`: `4.275`) y `regimen` en **4 siglas** (`AD`, `MP`, `PC`, `SA`).


In [5]:
# Auditoría paso a paso del filtro de RESERVA CONSUMIDA
c_estado = reservas["estado"].isin(["confirmada", "completada"])
c_no_cancel = reservas["fecha_cancelacion"].isna()
c_tarifa = (reservas["tarifa_noche"] > 0) & (reservas["tarifa_noche"] < 2000)
c_noches_hab = (reservas["noches"] > 0) & (reservas["num_habitaciones"] > 0)
c_periodo = reservas["fecha_entrada"].isna() | (reservas["fecha_entrada"] <= pd.Timestamp("2024-12-31"))
c_coherencia = (
    reservas["fecha_salida"].isna()
    | reservas["fecha_entrada"].isna()
    | (reservas["fecha_salida"] >= reservas["fecha_entrada"])
)

embudo = pd.DataFrame([
    {"paso": "0. Reservas únicas (tras deduplicar 1.500 duplicados)", "filas": len(reservas)},
    {"paso": "1. Sin fecha de cancelación (no canceladas de ayer)", "filas": int(c_no_cancel.sum())},
    {"paso": "2. Estado 'confirmada' o 'completada' + sin fecha_cancelacion", "filas": int((c_estado & c_no_cancel).sum())},
    {"paso": "3. + Tarifa válida (0 < tarifa_noche < 2.000 €)", "filas": int((c_estado & c_no_cancel & c_tarifa).sum())},
    {"paso": "4. + Noches > 0 y num_habitaciones > 0", "filas": int((c_estado & c_no_cancel & c_tarifa & c_noches_hab).sum())},
    {"paso": "5. + Entrada en el ejercicio (<= 2024-12-31)", "filas": int((c_estado & c_no_cancel & c_tarifa & c_noches_hab & c_periodo).sum())},
    {"paso": "6. + Salida coherente (fecha_salida >= fecha_entrada)", "filas": int((c_estado & c_no_cancel & c_tarifa & c_noches_hab & c_periodo & c_coherencia).sum())},
])
display(embudo)

consumidas = reservas[c_estado & c_no_cancel & c_tarifa & c_noches_hab & c_periodo & c_coherencia]
ingreso_total_grupo = (consumidas["noches"] * consumidas["tarifa_noche"]).sum()
noches_totales_grupo = consumidas["noches"].sum()
print(f"\nIngreso bruto total del grupo (consumidas) : {ingreso_total_grupo:,.2f} €")
print(f"Noches vendidas totales del grupo          : {noches_totales_grupo:,}")

# Exportamos las 6 tablas limpias a data/clean/
hoteles.to_csv(CLEAN_DIR / "hoteles.csv", index=False)
canales.to_csv(CLEAN_DIR / "canales.csv", index=False)
tipos_hab.to_csv(CLEAN_DIR / "tipos_habitacion.csv", index=False)
costes.to_csv(CLEAN_DIR / "costes_operativos.csv", index=False)
clientes.to_csv(CLEAN_DIR / "clientes.csv", index=False)
reservas.to_csv(CLEAN_DIR / "reservas.csv", index=False)
print("\nLas 6 tablas limpias han sido exportadas a 'data/clean/'.")


,paso,filas
0,0. Reservas únicas (tras deduplicar 1.500 dupl...,105917
1,1. Sin fecha de cancelación (no canceladas de ...,92791
2,2. Estado 'confirmada' o 'completada' + sin fe...,88516
3,3. + Tarifa válida (0 < tarifa_noche < 2.000 €),88504
4,4. + Noches > 0 y num_habitaciones > 0,88493
5,5. + Entrada en el ejercicio (<= 2024-12-31),88488
6,6. + Salida coherente (fecha_salida >= fecha_e...,88482



Ingreso bruto total del grupo (consumidas) : 23,040,000.00 €
Noches vendidas totales del grupo          : 218,000



Las 6 tablas limpias han sido exportadas a 'data/clean/'.


### Conclusión del Día 2: Definición de «reserva consumida» validada
* El embudo de filtrado muestra cómo cada condición elimina un tipo de ruido específico:
  * De las `92.791` reservas sin fecha de cancelación, quitar los `4.275` `no_show` deja `88.516`.
  * Quitar los `5` outliers de `99.999 €` y las `6` tarifas `<= 0 €` deja `88.504`.
  * Quitar las `6` filas con `noches <= 0` y las `5` filas con `num_habitaciones == 0` deja **`88.493` reservas** con importes y unidades válidas.
  * Finalmente, acotar la fecha de entrada a `<= 2024-12-31` (excluyendo `5` reservas anticipadas de 2027) y exigir `fecha_salida >= fecha_entrada` (excluyendo `6` estancias con salida anterior a la entrada) deja las **`88.482` estancias consumidas en 2024** que suman **`218.000` noches vendidas** y exactamente **$23.040.000,00	ext{ €}$ de ingreso de habitación clavados**.
* Las 6 tablas limpias ya están guardadas en `data/clean/` listas para consultarlas con SQL (DuckDB) en el **Día 3**.
